## 1. GitHub Actions for CI/CD

![](MLOPs_img/Docker-Container.png)

**Continuous Integration (CI)** involves regularly merging code changes into a shared repository. For ML, this means integrating new models, datasets, or code changes into the project repository to trigger automated testing.

**Continuous Delivery/Deployment (CD)** extends CI by automatically delivering or deploying the integrated code to production. In ML, this could involve deploying updated models or datasets to a production environment.

Tracking experiments is crucial for understanding model performance and changes over time. Tools like MLflow can be integrated into the CI/CD pipeline to monitor and log experiments.

Developer take action or trigger(s)(like push, commit, merge on master branch, pull request on master branch etc.) the CI/CD pipeline, which start the runner (workflow) to execute the defined steps in `.github/workflows/ci_cd.yml` (here, `ci_cd` is the custom name), such as building, testing, and deploying the ML model. It fetch & pull the latest code from given branch from given repository, run the workflow in given environment of given OS and send the output/feedback to the github repository back. If building and testing works successfully, means we can go ahead with CD to deploy the model to production. There are mannual review of CI and CT result before going to CD.

NOTE: In the setting > actions > general then Select to make CI/CT to work:
- **Actions permissions** should be **Allow all actions and reusable workflows**
- **Artifact and log retention** should be **90** (any depending upon need but high is best for record)
- **Approval for running fork pull request workflows from contributors** should be **Require approval for first-time contributors**
- **Workflow permissions** should be **Read and write permissions**
- **select/tick the option** **Allow GitHub Actions to create and approve pull requests**


#### Integrating CML with GitHub Actions for Version Control

CML (Continuous Machine Learning) helps you automatically train and evaluate machine learning models right in your pull/merge requests. It can also embed reports from the results, including plots and metrics.

`iterative` is owned by iterative that has developed `dvc` and `actions` is owned by GitHub, while `setup-cml` is to set up all the necessary requirements for machine learning.

#### VM Instance

To deploy the model to production, we need to set up a self-hosted runner on GCP (Google Cloud Platform) that can access GCS (Google Cloud Storage) for DVC. This allows us to run our CI/CD pipeline in an environment that has access to our data and can deploy our models effectively.

##### Step 1: Create the VM Instance (GCP)

1. Go to the [Google Cloud Console](https://console.cloud.google.com/).
2. Navigate to **Compute Engine** > **VM instances**.
3. Click **Create Instance**.
4. **Configure your VM:**
    -  **Name:** `github-runner-1`
    -  **Region/Zone:** Choose one close to you (e.g., `asia-south2`).
    -  **Machine type:** `e2-medium` (2 vCPU, 4GB RAM) is usually plenty for a basic runner.
    -  **Boot disk:** Ubuntu 22.04 LTS (standard for most GitHub Actions).
    -  **Identity and API access:** Under **Service Account**, select the "Compute Engine default service account" for now (or a custom one if you prefer higher security).
5. Click **Create**.

##### Step 2: Get the Runner Registration Commands

1. On GitHub, go to your **Repository Settings**.
2. In the left sidebar, click **Actions** > **Runners**.
3. Click **New self-hosted runner**.
4. Select **Linux** and **x64**.
5. **Stop here:** You will see a list of commands under "Download" and "Configure." Keep this tab open; you'll copy-paste these shortly.

##### Step 3: Install the Docker Engine on the VM
1. To update the package list and install Docker
```bash
sudo apt-get update
sudo apt-get install -y docker.io
```
2. Grant Permissions : By default, Docker requires `sudo` privileges. Your GitHub Action runner operates as a normal user and cannot type in a password for `sudo`. You need to add your user to the "docker" group so it can run Docker commands freely.
```bash
sudo usermod -aG docker $USER
```
Close and open the SSH terminal again to apply the group changes.

##### Step 4: Install the Runner on the VM

1. In the GCP Console VM list, click the **SSH** button next to your instance. A terminal window will pop up.
2. **Paste the "Download" commands** from the GitHub tab one by one. This downloads the runner software.
3. **Paste the "Configure" commands.**
    - When you run `./config.sh`, it will ask for a name. You can hit `Enter` for defaults.
    - It will ask for **labels**. If your workflow `.github/workflows/main.yml` uses `runs-on: self-hosted`, the default label is fine.
4. **Install as a background service** (Crucial so it doesn't stop when you close the SSH window):
```bash
sudo ./svc.sh install
sudo ./svc.sh start
```

##### Step 5: Access GCS for DVC (Authentication)

1. **On the VM**, verify the runner has access to GCP tools:
```bash
gcloud auth list
```
2. **Permissions:** Go to **IAM & Admin** in the GCP Console.
3. Find the Service Account used by your VM (usually `[project-number]-compute@developer.gserviceaccount.com`).
4. Click the **Edit** (pencil) icon and ensure it has the role **Storage Object Admin** so DVC can read/write to your GCS bucket.

#### Artifact Registry

##### Step 1: Create an Artifact Registry in GCP (Your ECR Equivalent)
In the Google Cloud Console, search for **Artifact Registry** in the top search bar.

1. Click **+ Create Repository**.
2. **Name**: Give it a name (e.g., `trip-duration-docker-image-hub`).
3. **Format**: Select **Docker**.
4. **Location type**: Choose Region and select the same region where your VM is located (e.g., `asia-south2`).
5. Click **Create**.

##### Step 2: Create a Service Account
Your GitHub Action (running on `ubuntu-latest`) needs permission to push images into this new registry. In GCP, we use a **Service Account** instead of an IAM User. The Service Account you created earlier `trip-duration-docker-image-hub-sa` only has permission for the Artifact Registry. It now needs permission to read your GCS bucket.

1. Go to **IAM & Admin** > **Service Accounts** in the GCP Console.
2. Click **+ Create Service Account**.
3. **Name**: Call it `trip-duration-docker-image-hub-sa` and click **Create and Continue**.
4. **Roles**: Grant it the `Artifact Registry Writer` (this allows it to push and pull images) and `Storage Object Viewer` roles (this gives it read-only access to GCS) using **Add Another Role**. Click **Done**.
5. Find your newly created service account in the list, click the three dots on the right, and select **Manage keys**.
6. Click **Add Key** > **Create new key** > Choose **JSON** > Click **Create**.
7. A JSON file will download to your computer.

##### Step 3: Add Secrets to GitHub
Go to your GitHub repository Settings > Secrets and variables > Actions and add the following repository secrets:

- `GCP_CREDENTIALS`: Open the downloaded JSON file in a text editor, copy the entire contents, and paste it here.
- `GCP_PROJECT_ID`: Your GCP Project ID (found on the GCP homepage).
- `GCP_REGION`: The region you chose (e.g., `asia-south2`).
- `GCP_AR_REPO_NAME`: The name of the Artifact Registry you created (e.g., `trip-duration-docker-image-hub`).

`docker ps` on VM Machine's Terminal confirming that the container is running successfully!
Use `docker attach trip-duration` to see the logs of the container in real-time.

##### Step 1:  Update GCP's default firewall rules
1. Go to **VPC network** -> **Firewall** in the GCP Console.
2. Click **Create Firewall Rule**.
3. **Name**: `allow-port-8080`
4. **Direction of traffic**: Ingress
5. **Action on match**: Allow
6. **Targets**: All instances in the network
7. **Source filter**: IPv4 ranges
8. **Source IPv4 ranges**: `0.0.0.0/0`
9. **Protocols and ports**: Select "Specified protocols and ports", check **TCP**, and enter `8080`.
10. Click **Create**.

##### Step 2: Find Your VM's External IP Address and visit the endpoint
To hit your endpoint, you need the public IP of the VM hosting the Docker container.
1. In the GCP Console, go to **Compute Engine** -> **VM instances**.
2. Locate your instance (the self-hosted runner).
3. Copy the IP address listed under the **External IP** column.
```text
http://<YOUR_VM_EXTERNAL_IP>:8080/
```

## 2. Containerization and Orchestration Philosophy

`Containerization` is a lightweight and efficient way to package, distribute, and run applications. It encapsulates an application and its dependencies into a single, standardized unit known as a **container**. Containerization ensures isolation from the underlying system and other containers. Containers provide consistency across various environments whether it's a developer's laptop, a testing environment, or a production server, making it easier to develop, deploy, and scale applications. setup/install necessary package, initialization issue, paths etc are taken care by the container. We can select which files to be included in the container and which not(/data, /dvclive etc.), out of the github repository. Container is faster and lighter than virtual machine and don't need to install OS again and again. In container one OS kernel run all the apps. No matter what OS you have in your device, just install docker and run the image of the container.

An `image` is a static snapshot of a container, while a `container` is a running instance of an image. An image contains the application code, libraries, and dependencies, while a container is the execution environment that runs the application based on the image.

`Orchestration` is the automated management, coordination, and deployment of multiple containers in a distributed environment. It addresses challenges such as scaling, load balancing, and service discovery.

Key aspects of orchestration:
- **Scaling:** Easily scale the number of containers to handle increased load or demand.
- **Service Discovery:** Automatically discover and connect containers to form a distributed application.
- **Load Balancing:** Distribute incoming traffic across multiple containers to ensure optimal performance and reliability.
- **Fault Tolerance:** Orchestration tools help in maintaining application availability by handling container failures and replacing them as needed.

![](MLOPs_img/containers-vs-virtual-machines.jpg)

#### [Docker](https://www.docker.com/get-started)

To create the necessary Docker assets like .dockerignore, compose.yaml, README.Docker.md and Dockerfile.
Fill the these details : `Python`, `3.13.5`, `8000` and `gunicorn 'app:app' --bind=0.0.0.0:8000`
```bash
docker init
```

To build or Containerization the image and run the container. Then visit `http://localhost:8000`.
Press Ctrl + C to exit the program from terminal
```bash
docker compose up --build
```

In order to push the image to the Docker Hub one need to make sure the user is login
```bash
docker login
```

#### format the image then push the image to the Docker Hub with the format 
docker tag image_name user_name/image_name:latest
```bash
docker tag python-docker vivekkumar7171/python-docker:latest
```

```bash
# To run a Docker container
docker run hello-world
```

```bash
# To stop Vmmem or WSL (Windows Subsystem for Linux), run the below command in PowerShell
wsl --shutdown
```

```bash
docker build -t mlops-app .
docker run -p 8000:8000 mlops-app
```

#### Creating a Multi-Container Application

Consider a scenario where you have a web application that communicates with a backend database. We'll create two containers, one for the web application and another for the database.

- Create a `docker-compose.yml` file:
```yaml
version: '3'
services:
  web:
    image: nginx:alpine
    ports:
      - "8080:80"
  db:
    image: mysql:5.7
    environment:
      MYSQL_ROOT_PASSWORD: example
      MYSQL_DATABASE: app_db
      MYSQL_USER: app_user
      MYSQL_PASSWORD: app_password

```

This `docker-compose.yml file` defines two services: `web` and `db`. The `web` service uses the Nginx image, while the `db` service uses the MySQL image. The `environment` section sets up MySQL with a root password and a database for our application.


- Run the multi-container application:
```bash
docker-compose up

```
This command starts both containers defined in the docker-compose.yml file.

#### Scaling with Docker Compose

Scaling is a crucial aspect of container orchestration. Docker Compose allows you to scale services easily.

- Update the `docker-compose.yml` file to include a service that can be scaled:

```yaml
version: '3'
services:
  web:
    image: nginx:alpine
    ports:
      - "8080:80"
  db:
    image: mysql:5.7
    environment:
      MYSQL_ROOT_PASSWORD: example
      MYSQL_DATABASE: app_db
      MYSQL_USER: app_user
      MYSQL_PASSWORD: app_password
  app_server:
    image: custom_app_image
    ports:
      - "5000:5000"

```

Here, we've added an `app_server` service. Replace `custom_app_image` with the actual image for your application server

- Scale the `app_server` service:

```bash
docker-compose up --scale app_server=3

```

This command scales the app_server service to three instances. You can adjust the number based on your requirements.

## 3. [AWS MLOps Services](https://aws.amazon.com/)

AWS (Amazon Web Services) extensive suite of services tailored for Machine Learning (ML) and MLOps (Machine Learning Operations) and uses of [**AWS CLI (Command Line Interface)**](https://docs.aws.amazon.com/cli/latest/userguide/cli-chap-welcome.html).

- [**Amazon SageMaker**](https://aws.amazon.com/sagemaker/) is a fully managed service that enables you to build, train, and deploy machine learning models at scale. It provides a Jupyter notebook interface for data exploration and model development.

- [**Amazon S3** ](https://aws.amazon.com/s3/) is a scalable object storage service, ideal for storing and retrieving any amount of data like your ML datasets and model artifacts. It enables Data Versioning in S3 to track changes in datasets over time.

- [**AWS Lambda**](https://aws.amazon.com/lambda/) is a serverless compute service that runs your code in response to events.

- [**AWS Step Functions**](https://aws.amazon.com/step-functions/) is a serverless orchestration service that coordinates workflows like automate the ML workflow.

- [**Amazon ECR (Elastic Container Registry)**](https://aws.amazon.com/ecr/) and [**Amazon ECS (Elastic Container Service)**](https://aws.amazon.com/ecs/) facilitate containerized deployment to store Docker containers with your ML models.

- [**AWS CodePipeline**](https://aws.amazon.com/codepipeline/) automates the ML model deployment process like the build, test, and deployment phases, while [**AWS CodeBuild**](https://aws.amazon.com/codebuild/) compiles your source code for building and testing ML code. Both combined help in CI/CD.

Task based on Profile:

- Data Scientist:
    data needs to be provided to the data scientist by the Engineering Manager to any platform like S3 etc.
    Problem Related/Specific Research: 
        Who has implemented it till now, 
        Research Paper published on the topic and its implementation
        algorithm that work in this problem and can be implemented in our Company scale.
    EDA: 
    Modeling:
        lastly push the code, notebook and scr folder in the github for ML Engineer. 

- ML Engineer:
    ML Engineer get project from the Data Scientist and ML Engineer convert it into an app/service. Also ML Engineer get technical specification (tech spec) from different teams that will use the app/service and evaluate if the tech spec is meet by the project.
    if tech spec requirements are fixed, then convert the project into a service that must have 24/7 availability (if one machine fails then other machine should start automatically), crash proof for large number of users etc.
    dockerize the service/app into image/docker/container and push to ECS then deploy and test the service/app on stage environment.

- MLOps Engineer:
    MLOps must know K8s, AWS and IAMs.
    push the service into production using his production access as as ML Engineer used to have access of only developer/stage environment.
    CI/CD:
    Monitoring: using Prometheus and Grafana.
    if any problem occur in the deployed service/app the only MLOps and ML Engineer will know and find the problem if the problem is related to code then ML Engineer will handle. if problem is related to deployment then MLOps will handle.

**Architecture Design**

- [AWS vs Free](https://excalidraw.com/#json=4ZsL2vE1dqRn1q027A8wf,K4kr706e4aZM7AKuH0i39w)

![AWS vs Free](MLOPs_img/aws_vs_free.svg)

- Small Teams

![Small Teams](MLOPs_img/Small_Teams.png)

- Medium Teams

![Medium Teams](MLOPs_img/Medium_Teams.png)

- Large Teams

![Large Teams](MLOPs_img/Large_Teams.png)

